# Evaluación de PL-BERT
Este notebook replica la lógica del antiguo script `eval.py` para evaluar un modelo PL-BERT usando Accelerate.

In [1]:
import os
import os.path as osp
import yaml
import torch
from torch.utils.data import DataLoader
from accelerate import Accelerator
from datasets import load_from_disk
from transformers import (
    BertTokenizer,
    BertConfig,
    BertForMaskedLM,
    DataCollatorForLanguageModeling
)
from tqdm import tqdm
import math

In [2]:
print("Cargando config.yml...")
config_path = "Configs/config.yml"
with open(config_path) as cfg_file:
    config = yaml.safe_load(cfg_file)

print("Cargando tokenizador desde Hugging Face...")
tokenizer = BertTokenizer.from_pretrained(config['dataset_params']['tokenizer_name'])

print("Cargando dataset de evaluación fonetizado...")
if 'test_data_folder' not in config:
    raise ValueError("Tu config.yml no tiene la clave 'test_data_folder'.")
test_dataset = load_from_disk(config['test_data_folder'])

def tokenize_function(examples):
    phoneme_strings = [' '.join(ph_list) for ph_list in examples['phonemes']]
    return tokenizer(
        phoneme_strings,
        padding=False,
        truncation=True,
        max_length=config['model_params']['max_position_embeddings'],
        return_special_tokens_mask=False,
    )

print("Tokenizando el dataset de evaluación en memoria...")
tokenized_test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=['phonemes']
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=config['dataset_params']['word_mask_prob']
)

Cargando config.yml...
Cargando tokenizador desde Hugging Face...
Cargando dataset de evaluación fonetizado...
Tokenizando el dataset de evaluación en memoria...
Cargando dataset de evaluación fonetizado...
Tokenizando el dataset de evaluación en memoria...


Map (num_proc=4):   0%|          | 0/2792 [00:00<?, ? examples/s]

In [5]:
def evaluate():
    accelerator = Accelerator()

    accelerator.print('Creando la arquitectura del modelo...')
    model_params = config['model_params']
    bert_config = BertConfig(
        vocab_size=len(tokenizer),
        hidden_size=model_params['hidden_size'],
        num_hidden_layers=model_params['num_hidden_layers'],
        num_attention_heads=model_params['num_attention_heads'],
        intermediate_size=model_params['intermediate_size'],
        max_position_embeddings=model_params['max_position_embeddings']
    )
    model = BertForMaskedLM(bert_config)

    log_dir = config['log_dir']
    try:
        ckpts = [f for f in os.listdir(log_dir) if f.startswith('step_')]
        iters = [int(f.split('_')[-1].split('.')[0]) for f in ckpts]
        last_step = sorted(iters)[-1]
        checkpoint_path = osp.join(log_dir, f'step_{last_step}.t7')

        accelerator.print(f"Cargando checkpoint final: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        model.load_state_dict(checkpoint['net'])
    except Exception as e:
        accelerator.print(f"No se pudo cargar ningún checkpoint en '{log_dir}'. Error: {e}")
        return

    batch_size = config['batch_size']
    test_loader = DataLoader(
        tokenized_test_dataset,
        batch_size=batch_size,
        collate_fn=data_collator,
        shuffle=False
    )

    model, test_loader = accelerator.prepare(model, test_loader)

    model.eval()
    total_loss = 0
    num_batches = 0

    accelerator.print('Iniciando evaluación...')
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Calculando Perplejidad'):
            outputs = model(**batch)
            loss = outputs.loss
            total_loss += accelerator.gather(loss).sum().item()
            num_batches += 1

    avg_loss = total_loss / (num_batches * accelerator.num_processes)
    perplexity = math.exp(avg_loss)

    print('='*30)
    print('Resultados de la evaluación')
    print('='*30)
    print(f'Modelo: {checkpoint_path}')
    print(f'Test Set: {config["test_data_folder"]}')
    print(f'Loss Promedio: {avg_loss:.4f}')
    print(f'Perplejidad: {perplexity:.4f}')
    print('='*30)

In [ ]:
# Ejecutar evaluación
evaluate()

Creando la arquitectura del modelo...
Cargando checkpoint final: Checkpoint_PhonemeBERT/step_25000.t7
Cargando checkpoint final: Checkpoint_PhonemeBERT/step_25000.t7
Iniciando evaluación...
Iniciando evaluación...


Calculando Perplejidad: 100%|██████████| 56/56 [00:09<00:00,  5.81it/s]

Resultados de la evaluación
Modelo: Checkpoint_PhonemeBERT/step_25000.t7
Test Set: corts_valencianes_test
Loss Promedio: 5.3431
Perplejidad: 209.1501


: 